# 06: Generate Thesis Figures
Create all publication-ready charts

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, os
from sklearn.metrics import confusion_matrix, f1_score
import warnings; warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 300; plt.rcParams['font.size'] = 11; plt.rcParams['axes.titlesize'] = 13; plt.rcParams['savefig.dpi'] = 300
os.makedirs('../results/figures', exist_ok=True)
print('=== NOTEBOOK 06: GENERATING FIGURES ===')
results = pd.read_csv('../results/model_comparison.csv')
predictions = pd.read_csv('../results/predictions.csv')
fi = pd.read_csv('../results/feature_importance.csv')
label_map = pd.read_csv('../data/processed/label_map.csv')
actual_classes = sorted(predictions['y_true'].unique())
class_names = [label_map.loc[label_map['encoded'] == i, 'technique'].values[0] for i in actual_classes]
model_cols = [c for c in predictions.columns if c != 'y_true']
# Fig 1: Model Comparison
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
for ax, metric, color in zip(axes, metrics, colors):
    bars = ax.bar(results['Model'], results[metric], color=color, edgecolor='black', linewidth=0.5)
    ax.set_ylim(0, 1.05); ax.set_ylabel(metric); ax.set_title(metric); ax.tick_params(axis='x', rotation=45)
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)
plt.suptitle('Model Performance Comparison', fontsize=15, y=1.02); plt.tight_layout(); plt.savefig('../results/figures/01_model_comparison.png'); plt.close(); print('Saved: 01_model_comparison.png')
# Fig 2: Confusion Matrices (already done in 04, but regenerate)
n_models = len(model_cols)
fig, axes = plt.subplots(2, 3, figsize=(15, 10)); axes = axes.flatten()
for idx, col in enumerate(model_cols):
    cm = confusion_matrix(predictions['y_true'], predictions[col], labels=actual_classes)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], xticklabels=class_names, yticklabels=class_names, cbar=False, square=True, linewidths=0.5)
    axes[idx].set_title(f'{col}'); axes[idx].set_xlabel('Predicted'); axes[idx].set_ylabel('True')
if n_models < 6: axes[5].axis('off')
plt.suptitle('Confusion Matrices', fontsize=15, y=1.02); plt.tight_layout(); plt.savefig('../results/figures/02_confusion_matrices.png'); plt.close(); print('Saved: 02_confusion_matrices.png')
# Fig 3: Feature Importance
fig, ax = plt.subplots(figsize=(10, 6))
top_n = min(15, len(fi)); top = fi.head(top_n); x = np.arange(top_n); w = 0.35
ax.barh(x - w/2, top['importance_rf'], w, label='Random Forest', color='#2E86AB')
ax.barh(x + w/2, top['importance_xgb'], w, label='XGBoost', color='#F18F01')
ax.set_yticks(x); ax.set_yticklabels(top['feature']); ax.invert_yaxis(); ax.set_xlabel('Importance'); ax.set_title('Top Feature Importance'); ax.legend()
plt.tight_layout(); plt.savefig('../results/figures/03_feature_importance.png'); plt.close(); print('Saved: 03_feature_importance.png')
# Fig 4: Per-Technique F1
f1_data = []
for col in model_cols:
    f1s = f1_score(predictions['y_true'], predictions[col], labels=actual_classes, average=None, zero_division=0)
    f1_data.append(f1s)
f1_matrix = np.array(f1_data)
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(f1_matrix, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1, xticklabels=class_names, yticklabels=model_cols, ax=ax, linewidths=0.5, linecolor='white', cbar_kws={'label': 'F1'})
ax.set_title('Per-Technique F1 Score'); ax.set_xlabel('ATT&CK Technique')
plt.tight_layout(); plt.savefig('../results/figures/04_f1_summary.png'); plt.close(); print('Saved: 04_f1_summary.png')
# Fig 5: CV Box Plot
cv_results = pd.read_csv('../results/cv_results.csv') if os.path.exists('../results/cv_results.csv') else None
if cv_results is not None:
    cv_melted = cv_results.melt(id_vars=['Model'], var_name='Fold', value_name='F1')
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=cv_melted, x='Model', y='F1', palette='Set2', ax=ax)
    sns.stripplot(data=cv_melted, x='Model', y='F1', color='black', size=6, ax=ax)
    ax.set_ylim(0, 1.05); ax.set_title('5-Fold Stratified CV F1 Scores'); ax.tick_params(axis='x', rotation=45)
    plt.tight_layout(); plt.savefig('../results/figures/05_cv_boxplot.png'); plt.close(); print('Saved: 05_cv_boxplot.png')
# Fig 6: Class Distribution
fig, ax = plt.subplots(figsize=(10, 5))
counts = predictions['y_true'].value_counts().sort_index()
bars = ax.bar([class_names[actual_classes.index(c)] for c in counts.index], counts.values, color='#2E86AB', edgecolor='black')
ax.set_ylabel('Samples'); ax.set_title('Test Set Class Distribution'); ax.tick_params(axis='x', rotation=45)
for bar in bars:
    h = bar.get_height()
    ax.annotate(f'{int(h)}', xy=(bar.get_x() + bar.get_width()/2, h), xytext=(0, 3), textcoords='offset points', ha='center')
plt.tight_layout(); plt.savefig('../results/figures/06_class_distribution.png'); plt.close(); print('Saved: 06_class_distribution.png')
print('\n=== ALL FIGURES GENERATED ===')
